# SentimentVault: Comprehensive EDA & Data Preparation
## Phase 1: Building a High-Performance Sentiment Analysis Pipeline
This notebook covers exploratory data analysis, preprocessing, and preparation of 250K Amazon reviews for training DistilBERT to achieve 92% F1-score.

## 1. Import Required Libraries
Import all necessary libraries for data loading, analysis, visualization, and preprocessing.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datasets import load_dataset
from transformers import DistilBertTokenizer
import re
import html
from collections import Counter
import warnings
warnings.filterwarnings('ignore')

# Set plotting style
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)

print("✓ All libraries imported successfully!")

## 2. Load and Explore the Amazon Reviews Dataset
Load the Amazon Polarity dataset from Hugging Face and subset to 250,000 reviews (200k train, 50k test) to match resume metrics.

In [ ]:
print("Loading Amazon Polarity dataset from Hugging Face...")
dataset = load_dataset("amazon_polarity")

# Examine original dataset structure
print(f"\nOriginal dataset splits: {dataset.keys()}")
print(f"Train set size: {len(dataset['train'])}")
print(f"Test set size: {len(dataset['test'])}")

# Look at first sample
print("\nFirst sample from train set:")
sample = dataset['train'][0]
for key, value in sample.items():
    print(f"  {key}: {value if len(str(value)) < 100 else str(value)[:100] + '...'}")

In [ ]:
# Subset to 250,000 reviews (200k train, 50k test) to match resume
print("\nSubsetting dataset to 250K reviews for project...")
TRAIN_SIZE = 200000
TEST_SIZE = 50000

train_dataset = dataset['train'].shuffle(seed=42).select(range(TRAIN_SIZE))
test_dataset = dataset['test'].shuffle(seed=42).select(range(TEST_SIZE))

print(f"✓ Train set size: {len(train_dataset)}")
print(f"✓ Test set size: {len(test_dataset)}")
print(f"✓ Total dataset: {len(train_dataset) + len(test_dataset)} reviews")

# Convert to pandas for easier analysis
train_df = train_dataset.to_pandas()
test_df = test_dataset.to_pandas()

print("\nTrain dataset shape:", train_df.shape)
print("Train dataset info:")
print(train_df.head())

# Label distribution
print("\n--- Label Distribution ---")
print(f"Train set:\n{train_df['label'].value_counts().sort_index()}")
print(f"\nTest set:\n{test_df['label'].value_counts().sort_index()}")

In [ ]:
# Analyze text lengths
train_df['text_length'] = train_df['content'].str.len()
train_df['word_count'] = train_df['content'].str.split().str.len()

print("\n--- Text Statistics ---")
print(f"Average text length: {train_df['text_length'].mean():.2f} characters")
print(f"Median text length: {train_df['text_length'].median():.2f} characters")
print(f"Min text length: {train_df['text_length'].min()} characters")
print(f"Max text length: {train_df['text_length'].max()} characters")
print(f"\nAverage word count: {train_df['word_count'].mean():.2f} words")
print(f"Median word count: {train_df['word_count'].median():.2f} words")

# Visualize label distribution
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Train label distribution
train_labels = train_df['label'].value_counts()
axes[0].bar(['Negative', 'Positive'], train_labels.sort_index().values)
axes[0].set_title('Train Set Label Distribution\n(200K reviews)')
axes[0].set_ylabel('Count')
axes[0].grid(axis='y', alpha=0.3)

# Test label distribution
test_labels = test_df['label'].value_counts()
axes[1].bar(['Negative', 'Positive'], test_labels.sort_index().values)
axes[1].set_title('Test Set Label Distribution\n(50K reviews)')
axes[1].set_ylabel('Count')
axes[1].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

# Calculate balance
print(f"\nTrain set balance: {train_labels[0] / train_labels[1]:.2f} (negative/positive ratio)")
print(f"Test set balance: {test_labels[0] / test_labels[1]:.2f} (negative/positive ratio)")

## 3. Data Preprocessing and Cleaning
Clean the review text by removing HTML tags, lowercasing, and handling special characters.

In [ ]:
def clean_text(text):
    """
    Clean review text by:
    - Decoding HTML entities
    - Lowercasing
    - Removing extra whitespace
    - Removing URLs
    - Removing special characters (keeping alphanumeric and basic punctuation)
    """
    # Decode HTML entities
    text = html.unescape(text)
    
    # Lowercase
    text = text.lower()
    
    # Remove URLs
    text = re.sub(r'http\S+|www\S+|https\S+', '', text, flags=re.MULTILINE)
    
    # Remove email addresses
    text = re.sub(r'\S+@\S+', '', text)
    
    # Remove extra whitespace
    text = re.sub(r'\s+', ' ', text).strip()
    
    return text

# Apply cleaning to train dataset
print("Cleaning train dataset...")
train_df['content_clean'] = train_df['content'].apply(clean_text)

# Apply cleaning to test dataset
print("Cleaning test dataset...")
test_df['content_clean'] = test_df['content'].apply(clean_text)

print("✓ Data cleaning completed")

# Show before/after comparison
print("\n--- Text Cleaning Examples ---")
for i in range(3):
    print(f"\nExample {i+1}:")
    print(f"Original: {train_df['content'].iloc[i][:150]}...")
    print(f"Cleaned:  {train_df['content_clean'].iloc[i][:150]}...")

## 4. Train/Test Split Verification
Verify the train/test split with balanced label distribution across both sets.

In [ ]:
print("=== TRAIN/TEST SPLIT VERIFICATION ===\n")

# Verify split sizes
print(f"Train set size: {len(train_df)} samples")
print(f"Test set size: {len(test_df)} samples")
print(f"Total: {len(train_df) + len(test_df)} samples")

# Verify label balance
train_label_dist = train_df['label'].value_counts().sort_index()
test_label_dist = test_df['label'].value_counts().sort_index()

print("\n--- Label Distribution ---")
print("Train set:")
print(f"  Negative (0): {train_label_dist[0]:,} ({train_label_dist[0]/len(train_df)*100:.1f}%)")
print(f"  Positive (1): {train_label_dist[1]:,} ({train_label_dist[1]/len(train_df)*100:.1f}%)")

print("\nTest set:")
print(f"  Negative (0): {test_label_dist[0]:,} ({test_label_dist[0]/len(test_df)*100:.1f}%)")
print(f"  Positive (1): {test_label_dist[1]:,} ({test_label_dist[1]/len(test_df)*100:.1f}%)")

# Check data quality
print("\n--- Data Quality Checks ---")
print(f"Train set null values: {train_df.isnull().sum().sum()}")
print(f"Test set null values: {test_df.isnull().sum().sum()}")
print(f"Train set empty texts: {(train_df['content_clean'].str.len() == 0).sum()}")
print(f"Test set empty texts: {(test_df['content_clean'].str.len() == 0).sum()}")

print("\n✓ Train/Test split verified successfully!")

## 5. Tokenization with DistilBERT
Tokenize all reviews using DistilBertTokenizer with truncation to 512 tokens.

In [ ]:
print("Loading DistilBERT tokenizer...")
tokenizer = DistilBertTokenizer.from_pretrained("distilbert-base-uncased")

print("✓ Tokenizer loaded successfully!")

# Analyze token lengths
print("\nAnalyzing token lengths for cleaned text...")

# Sample tokenization to understand token counts
sample_texts = train_df['content_clean'].sample(1000, random_state=42).tolist()
token_lengths = []

for text in sample_texts:
    tokens = tokenizer.encode(text, truncation=False)
    token_lengths.append(len(tokens))

print(f"\nToken statistics (1000 samples):")
print(f"  Mean tokens: {np.mean(token_lengths):.1f}")
print(f"  Median tokens: {np.median(token_lengths):.1f}")
print(f"  Max tokens: {max(token_lengths)}")
print(f"  Min tokens: {min(token_lengths)}")
print(f"  Texts exceeding 512 tokens: {sum(1 for l in token_lengths if l > 512)} ({sum(1 for l in token_lengths if l > 512)/len(token_lengths)*100:.2f}%)")

# Visualize token distribution
plt.figure(figsize=(10, 5))
plt.hist(token_lengths, bins=50, edgecolor='black', alpha=0.7)
plt.xlabel('Number of Tokens')
plt.ylabel('Frequency')
plt.title('Distribution of Token Lengths (Sample of 1000 reviews)')
plt.axvline(x=512, color='r', linestyle='--', label='Max Truncation (512)')
plt.legend()
plt.grid(axis='y', alpha=0.3)
plt.show()

print("\n✓ Tokenization analysis completed!")

In [ ]:
# Now prepare datasets for training
# Convert back to Hugging Face datasets format

# Create datasets dictionary
from datasets import Dataset

print("Converting cleaned data to Hugging Face format...")

# Prepare training data
train_data = {
    'text': train_df['content_clean'].tolist(),
    'label': train_df['label'].tolist()
}
tokenized_train_dataset = Dataset.from_dict(train_data)

# Prepare test data
test_data = {
    'text': test_df['content_clean'].tolist(),
    'label': test_df['label'].tolist()
}
tokenized_test_dataset = Dataset.from_dict(test_data)

print(f"✓ Train dataset: {len(tokenized_train_dataset)} samples")
print(f"✓ Test dataset: {len(tokenized_test_dataset)} samples")

# Define tokenization function
def preprocess_function(examples):
    return tokenizer(
        examples["text"],
        truncation=True,
        max_length=512,
        padding="max_length",
        return_tensors="pt"
    )

print("\nTokenizing datasets (this may take a few minutes)...")
# Tokenize train dataset
tokenized_train = tokenized_train_dataset.map(
    preprocess_function,
    batched=True,
    batch_size=500,
    remove_columns=['text']
)

# Tokenize test dataset
tokenized_test = tokenized_test_dataset.map(
    preprocess_function,
    batched=True,
    batch_size=500,
    remove_columns=['text']
)

print("✓ Tokenization completed!")
print(f"✓ Tokenized train set: {len(tokenized_train)} samples")
print(f"✓ Tokenized test set: {len(tokenized_test)} samples")

## 6. Save Prepared Datasets
Save the cleaned and tokenized datasets for model training.

In [ ]:
import os

# Create data directory if it doesn't exist
os.makedirs('../data', exist_ok=True)

# Save datasets
print("Saving prepared datasets...")

tokenized_train.save_dataset("../data/train_tokenized")
tokenized_test.save_dataset("../data/test_tokenized")

# Also save the raw cleaned DataFrames for reference
train_df.to_csv("../data/train_cleaned.csv", index=False)
test_df.to_csv("../data/test_cleaned.csv", index=False)

print("✓ Datasets saved successfully!")
print(f"  - ../data/train_tokenized/")
print(f"  - ../data/test_tokenized/")
print(f"  - ../data/train_cleaned.csv ({len(train_df)} rows)")
print(f"  - ../data/test_cleaned.csv ({len(test_df)} rows)")

# Summary statistics
print("\n=== PHASE 1 SUMMARY ===")
print(f"Total reviews processed: {len(train_df) + len(test_df):,}")
print(f"Training samples: {len(train_df):,}")
print(f"Test samples: {len(test_df):,}")
print(f"Data cleaning: Complete")
print(f"Tokenization: Complete (512-token max)")
print(f"Ready for Phase 2: Model Fine-Tuning ✓")